[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pflahert/hd-stats-workshop/blob/main/notebooks/lab1.ipynb)

# Lab 1: The Gene That Wasn't There
## The False Discovery Problem

**Day 1, 9:00–10:00** | Learning Outcomes: LO1, LO3

This lab applies **Lecture 1** (hypothesis testing in high dimensions): the $p$-value histogram diagnostic, Bonferroni / FWER, and the Benjamini–Hochberg procedure for FDR control. You'll run 3,051 hypothesis tests on the Golub leukemia dataset, read the resulting histogram, compare the discovery counts under three rejection rules, and use AI to extend the analysis with a power-curve simulation.

By the end of this lab you will:
1. Run 3,051 hypothesis tests on the Golub leukemia dataset.
2. Read a $p$-value histogram and know what it tells you about signal, $\pi_0$, and pipeline behavior.
3. Compare the discovery counts under no correction, Bonferroni, and BH at two FDR levels.
4. Use AI to extend the analysis (single-shot and multi-turn) and critique the output against a named checklist.

---

**How this lab uses AI**

This lab integrates AI at several steps, not just at the end. Two kinds of AI exercises appear:

- **Single-shot critiques** — you prompt, paste the response, and evaluate it against a named checklist (Exercise 4.3, Section 7.2).
- **Multi-turn conversations** — you prompt, critique the response, write a follow-up prompt that addresses the specific weakness you flagged, and continue *in the same chat session* so the AI can revise its earlier answer. *At least one true multi-turn exercise per lab is required.* In this lab, **Exercise 5.3 (Bonferroni vs BH)** is the multi-turn one.

Use any AI assistant (the [UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), Claude, ChatGPT, or another). Note which AI you used for each prompt — different models behave differently, and the differences are part of the lesson.

---

The Golub dataset stays with us for the entire workshop. This is the first time you'll touch it; by the end of Day 2 you'll have tested it, reduced it, and built predictive models from it.


In [ ]:
# Setup — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Workshop convention: deterministic seed for any simulation in this notebook.
np.random.seed(2026)

# Surface RuntimeWarnings (e.g., zero-variance genes producing NaN p-values)
# rather than hiding them — these are exactly the failures students need to see.

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('default')

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12


---
## Part 1 — Meet the Data

The **Golub** dataset is a landmark microarray study from Golub et al. (1999). It measures expression levels of **3,051 genes** across **72 patients** diagnosed with either **ALL** (acute lymphoblastic leukemia) or **AML** (acute myeloid leukemia).

This is a classic high-dimensional setting: $p = 3{,}051$ features and $n = 72$ observations. The ratio $p/n \approx 42$ means we have roughly 42 times more variables than samples. Every method we study in this workshop exists because of this imbalance.

In [ ]:
# Load the Golub expression matrix and metadata directly from the workshop repo.
expr_url = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/v2026-spring-data/data/golub_expression.csv"
meta_url = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/v2026-spring-data/data/golub_metadata.csv"

# The CSV stores genes as rows and samples as columns; we want samples as rows for the t-tests below.
expr_genes_x_samples = pd.read_csv(expr_url, index_col=0)
meta = pd.read_csv(meta_url)

# Shape assertions catch any silent upstream change.
assert expr_genes_x_samples.shape == (3051, 72), \
    f"Unexpected expression shape: {expr_genes_x_samples.shape} (expected (3051, 72))"
assert list(expr_genes_x_samples.columns) == list(meta["sample_id"]), \
    "Sample IDs do not match between expression and metadata."

# Transpose to samples (rows) x genes (columns) for the per-gene analyses that follow.
expr = expr_genes_x_samples.T
expr.index.name = "sample_id"
labels = meta["subtype"].values

print(f"Expression matrix shape: {expr.shape}")
print(f"Samples (n): {expr.shape[0]}")
print(f"Genes (p): {expr.shape[1]}")

label_counts = pd.Series(labels).value_counts()
print(f"ALL samples: {label_counts.get('ALL', 0)}")
print(f"AML samples: {label_counts.get('AML', 0)}")
print(f"Ratio p/n: {expr.shape[1] / expr.shape[0]:.1f}")

# Create sample masks for the two groups
all_mask = labels == 'ALL'
aml_mask = labels == 'AML'

print(f"\nFirst few genes: {expr.columns[:5].tolist()}")


### Exercise 1.1

Look at the ratio $p/n$. You have roughly 42 times more variables than observations. If you tried ordinary linear regression with all 3,051 genes as predictors, what would happen? (We'll return to this on Day 2.)

**YOUR ANSWER:**



---
## Part 2 — One Gene, One Test

Before we test thousands of genes at once, let's start with familiar territory: one gene, one hypothesis test. We pick a single gene, compare its expression between ALL and AML patients, and ask whether the difference is statistically significant.

In [ ]:
# Pick the first gene
gene_name = expr.columns[0]
gene_expr = expr[gene_name]

all_expr = gene_expr[all_mask].values
aml_expr = gene_expr[aml_mask].values

# Boxplot comparing ALL vs AML expression
fig, ax = plt.subplots(figsize=(6, 5))
bp = ax.boxplot([all_expr, aml_expr], tick_labels=['ALL', 'AML'],
                patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#4C72B0')
bp['boxes'][1].set_facecolor('#DD8452')
ax.set_ylabel('Expression level')
ax.set_title(f'Gene: {gene_name}')

# Two-sample Welch t-test (no equal-variance assumption)
t_stat, p_val = stats.ttest_ind(all_expr, aml_expr, equal_var=False)
ax.text(0.95, 0.95, f't = {t_stat:.3f}\np = {p_val:.4f}',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

print(f"Gene: {gene_name}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_val:.4f}")


One gene, one test, one p-value. If the p-value is below 0.05, you'd call the gene significant. But you don't have one gene. You have 3,051.

---
## Part 3 — 3,051 Tests

Now we do what a real genomics study does: test every gene for differential expression between ALL and AML leukemia.

In [ ]:
# Run a Welch t-test for every gene in one vectorized call.
#
# First: drop genes whose expression is (nearly) constant within either group.
# These genes carry no information for differential expression — scipy's t-test
# will flag a "precision loss in moment calculation" RuntimeWarning on them and
# the resulting t-statistic / p-value would be unreliable. Filtering them is a
# standard preprocessing step in real RNA-seq / microarray pipelines.
gene_names_full = expr.columns.tolist()
all_block_full = expr.loc[all_mask].values.astype(float)   # 47 x 3051
aml_block_full = expr.loc[aml_mask].values.astype(float)   # 25 x 3051

MIN_GROUP_VAR = 1e-8
informative = (all_block_full.var(axis=0) > MIN_GROUP_VAR) & \
              (aml_block_full.var(axis=0) > MIN_GROUP_VAR)
n_dropped = int((~informative).sum())
if n_dropped:
    print(f"Dropped {n_dropped} gene(s) with near-constant expression in one or both groups.")

all_block = all_block_full[:, informative]
aml_block = aml_block_full[:, informative]
gene_names = [g for g, keep in zip(gene_names_full, informative) if keep]
n_genes = len(gene_names)

# ttest_ind with axis=0 runs the per-column t-test in one numpy operation.
t_stats, pvals = stats.ttest_ind(all_block, aml_block, equal_var=False, axis=0)

# Defensive: any remaining non-finite p-values would indicate a numerical issue.
n_nan = int((~np.isfinite(pvals)).sum())
if n_nan:
    print(f"WARNING: {n_nan} gene(s) still produced non-finite p-values after filtering.")

alpha = 0.05
n_significant = int((pvals < alpha).sum())
expected_fp = n_genes * alpha

print(f"Total genes tested: {n_genes}")
print(f"Significant at alpha = 0.05 (no correction): {n_significant}")
print(f"Expected false positives if nothing were real: {expected_fp:.0f}")


### Exercise 3.1

How many genes were called significant? How many false positives would you *expect* if none of the genes were truly differentially expressed? Where does that expected number come from? (Hint: $3{,}051 \times 0.05 = ?$)

**YOUR ANSWER:**



---
## Part 4 — The P-Value Histogram

The $p$-value histogram is the first diagnostic to consult before any correction is applied. Three readings come straight off the picture:

1. Whether there is any signal at all (spike near zero).
2. Roughly what fraction of tests are true nulls ($\pi_0$, estimated from the flat region's height relative to the uniform-null line).
3. Whether the analysis is well-behaved (an unexpected shape — bumps in the middle, a rising right tail, a wholesale shift — flags a misspecified test before any rejection rule is applied).

In [ ]:
# P-value histogram
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(pvals, bins=100, edgecolor='white', color='#4C72B0', alpha=0.8)

# Expected height under uniform null: n_genes / n_bins
expected_height = n_genes / 100
ax.axhline(y=expected_height, color='red', linestyle='--', linewidth=2,
           label=f'Expected under null ({expected_height:.0f} per bin)')

ax.set_xlabel('P-value')
ax.set_ylabel('Frequency')
ax.set_title('P-value histogram: 3,051 gene-level t-tests')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 4.1

Describe what you see. Where is the signal? Where is the noise?

**YOUR ANSWER:**

### Exercise 4.2

Estimate $\pi_0$ (the proportion of true nulls) by eye. Look at the flat region of the histogram and compare its height to the *expected* height under the uniform null (the dashed line).

**YOUR ANSWER:**


### Exercise 4.3 — AI interprets the histogram (single-shot)

Read off the figure above the approximate *ratio* between the height of the leftmost spike (the $[0, 0.05]$ bin) and the dashed uniform-null line. Use that number in the prompt below.

> "Reply in three short sentences only (no .docx file). I generated a *p*-value histogram from 3,051 two-sample Welch $t$-tests comparing two leukemia subtypes (ALL vs AML) on the Golub microarray dataset. The histogram has a tall spike near 0 (approximately **[YOUR RATIO]×** the uniform-null density) and a roughly flat region from about $p = 0.3$ to $p = 1$ at a height somewhat below the uniform-null density. Answer in three sentences: (1) is there real signal in this dataset? (2) approximately what fraction of genes are truly null? (3) what does the *shape* of the flat region tell me about whether my analysis pipeline is well-behaved?"

**AI's response (paste here):**


**Your critique** — evaluate against:

| Criterion | Pass / Fail / Partial | Notes |
|---|---|---|
| Did the AI correctly identify the spike near zero as real signal? | | |
| Did it estimate $\hat\pi_0$ from the flat-region height? | | |
| Did it interpret the flat region as the analysis being well-behaved (i.e., produces a uniform null when it should)? | | |
| Did it avoid the common error of treating spike size as a $p$-value? | | |

**One-sentence overall assessment** — did the AI's reading match your own visual reading?

### Verify with a pure-null simulation

To understand what "no signal" looks like, we'll simulate 3,051 pure-null t-tests (both groups drawn from the same distribution) and compare.

In [ ]:
# Null simulation: 3,051 tests where both groups come from N(0,1)
# Use the actual group sizes from the Golub dataset (47 ALL, 25 AML)
np.random.seed(2026)
n_all = int(all_mask.sum())   # 47
n_aml = int(aml_mask.sum())   # 25
null_pvals = np.zeros(n_genes)

for i in range(n_genes):
    x = np.random.normal(0, 1, n_all)
    y = np.random.normal(0, 1, n_aml)
    _, null_pvals[i] = stats.ttest_ind(x, y, equal_var=False)

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(null_pvals, bins=100, edgecolor='white', color='#999999', alpha=0.8)
axes[0].axhline(y=expected_height, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('P-value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Simulated null (no real signal)')

axes[1].hist(pvals, bins=100, edgecolor='white', color='#4C72B0', alpha=0.8)
axes[1].axhline(y=expected_height, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('P-value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Real data (Golub dataset)')

plt.tight_layout()
plt.show()

### Exercise 4.4

How many "significant" genes appear in the null simulation? Is it close to 5% of 3,051?

**YOUR ANSWER:**



In [ ]:
# Count false positives in the null simulation
null_sig = np.sum(null_pvals < 0.05)
print(f"False positives in null simulation: {null_sig}")
print(f"Expected (5% of {n_genes}): {n_genes * 0.05:.0f}")
print(f"Actual percentage: {100 * null_sig / n_genes:.1f}%")

---
## Part 5 — Bonferroni vs. Benjamini-Hochberg

We have thousands of significant p-values, but we know many could be false positives. Two classic corrections control different error rates:

- **Bonferroni** controls the family-wise error rate (FWER) — the probability of *any* false positive. Divide $\alpha$ by the number of tests.
- **Benjamini-Hochberg** controls the false discovery rate (FDR) — the expected fraction of false positives among declared discoveries. Allows more discoveries by tolerating a controlled fraction of false positives.

In [ ]:
# Bonferroni correction
bonf_threshold = 0.05 / n_genes
n_bonf = np.sum(pvals < bonf_threshold)

print(f"Bonferroni threshold: {bonf_threshold:.2e}")
print(f"Genes surviving Bonferroni: {n_bonf}")

In [ ]:
# Benjamini-Hochberg correction
reject_bh_05, pvals_bh_05, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
reject_bh_10, pvals_bh_10, _, _ = multipletests(pvals, alpha=0.10, method='fdr_bh')

n_bh_05 = np.sum(reject_bh_05)
n_bh_10 = np.sum(reject_bh_10)

print(f"Genes significant at FDR \u2264 0.05 (BH): {n_bh_05}")
print(f"Genes significant at FDR \u2264 0.10 (BH): {n_bh_10}")

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'Method': ['No correction (α = 0.05)', 'Bonferroni (α = 0.05)',
               'BH (FDR ≤ 0.05)', 'BH (FDR ≤ 0.10)'],
    'Genes called significant': [np.sum(pvals < 0.05), n_bonf, n_bh_05, n_bh_10]
})
print(comparison.to_string(index=False))

### Exercise 5.1

You're a biologist with funding to follow up 20 genes with qPCR validation. Which gene list would you use and why?

**YOUR ANSWER:**

### Exercise 5.2

A collaborator says "Just use Bonferroni — it's the safest." Write 2–3 sentences explaining why that might not be the best choice here.

**YOUR ANSWER:**


### Exercise 5.3 — Multi-turn: why does BH find more genes than Bonferroni? *(required multi-turn)*

This is the multi-turn AI exercise for Lab 1. Three turns of model output in the **same chat session** so the AI can revise its earlier answer — *don't start a fresh conversation between turns*.

Before you begin, look at the comparison table you printed in the previous cell and record the actual Bonferroni count and the BH(FDR≤0.05) count. You will use *your numbers from this run*, not pre-supplied placeholders.

| Method | Genes called significant (this run) |
|---|---|
| Bonferroni ($\alpha = 0.05$) | |
| BH (FDR $\leq 0.05$) | |

**Turn 1 — initial prompt.** Ask the AI, filling in your own numbers:

> "On the Golub leukemia dataset (3,051 two-sample Welch $t$-tests), the Bonferroni correction at $\alpha = 0.05$ gives me about **[YOUR BONF COUNT]** significant genes; the Benjamini–Hochberg procedure at FDR $\leq 0.05$ gives me about **[YOUR BH COUNT]**. Why does BH give so many more discoveries than Bonferroni? Two short paragraphs."

**AI's response (Turn 1, paste here):**


**Your critique (write your own — do not copy the bullets below verbatim).** Common failure modes to look for:

- The AI states "BH controls FDR, Bonferroni controls FWER" but doesn't *explain why that produces more discoveries.*
- The AI confuses FWER with FDR (uses the wrong definition for one of them).
- The AI explains the BH threshold formula ($k\alpha/m$) but doesn't connect it to "BH effectively uses a less stringent per-test threshold when $k$ is large."
- The AI does not mention the factor $m_0/m$ — i.e., that BH's bound depends on the proportion of true nulls, while Bonferroni's bound does not.
- The AI does not mention that $\pi_0 < 1$ on this dataset (visible from the histogram), so Bonferroni's $\alpha/m$ threshold is unnecessarily strict.

**YOUR CRITIQUE OF THE TURN-1 RESPONSE:**


**Turn 2 — your follow-up prompt.** Based on the specific weaknesses you flagged, write a prompt in your own words that asks the AI to revise its answer. *In the same chat session*, send it. The point of "multi-turn" is that the AI can see and revise its prior response — so paste no canned text; write the follow-up that targets what *you* found missing.

**YOUR TURN-2 PROMPT:**

**AI's response (Turn 2):**


**Turn 3 — final ask.** In the same conversation, ask the AI for one short paragraph that you could paste into a methods section. State explicitly what you want it to include (e.g., the connection between $\pi_0 < 1$ and the gap between BH and Bonferroni discovery counts on this data).

**YOUR TURN-3 PROMPT:**

**AI's response (Turn 3):**


**Final assessment.** Compare the three turns. Did Turn 2's targeted critique actually produce a better Turn-3 paragraph than Turn 1's first attempt? Note one thing the AI revised between turns and one thing it did not.

---
## Part 6 — What Goes Wrong Without Correction

The multiple testing problem doesn't just affect individual studies. When combined with publication bias and the winner's curse, it creates systematic distortions in the scientific literature. Let's simulate both.

In [ ]:
# Publication bias simulation
# 1,000 independent labs each test ONE null gene (no real effect)
np.random.seed(2026)
n_labs = 1000
n_per_group = 30
lab_pvals = np.zeros(n_labs)

for i in range(n_labs):
    x = np.random.normal(0, 1, n_per_group)
    y = np.random.normal(0, 1, n_per_group)  # same distribution — no real effect
    _, lab_pvals[i] = stats.ttest_ind(x, y, equal_var=False)

# Only "significant" results get published
published = lab_pvals < 0.05
n_published = int(np.sum(published))

print(f"Labs that ran the study: {n_labs}")
print(f"Labs that found p < 0.05 (published): {n_published}")
print(f"Fraction published: {n_published / n_labs:.1%}")
print()
print("In the published subset:")
print(f"  100% of the published papers report a 'significant' effect (by construction —")
print(f"  only significant results were published).")
print(f"  A reader of the literature sees only this subset and concludes the effect is real.")
print(f"  But the true effect is ZERO.")


### Exercise 6.1

What fraction of all studies found a significant effect? What fraction of *published* studies show a significant effect? This gap is publication bias. Explain in your own words why this is dangerous.

**YOUR ANSWER:**



In [ ]:
# Winner's curse simulation
# 1,000 studies with a REAL but small effect (d = 0.3)
np.random.seed(2026)
true_effect = 0.3
n_studies = 1000
n_per_group = 30
observed_effects = np.zeros(n_studies)
study_pvals = np.zeros(n_studies)

for i in range(n_studies):
    x = np.random.normal(0, 1, n_per_group)
    y = np.random.normal(true_effect, 1, n_per_group)
    observed_effects[i] = np.mean(y) - np.mean(x)
    _, study_pvals[i] = stats.ttest_ind(x, y, equal_var=False)

# Only significant studies get published
published_mask = study_pvals < 0.05
published_effects = observed_effects[published_mask]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(published_effects, bins=30, edgecolor='white', color='#DD8452', alpha=0.8,
        label=f'Published effect sizes (n={len(published_effects)})')
ax.axvline(x=true_effect, color='green', linewidth=2, linestyle='-',
           label=f'True effect (d = {true_effect})')
ax.axvline(x=np.mean(published_effects), color='red', linewidth=2, linestyle='--',
           label=f'Mean published effect (d = {np.mean(published_effects):.2f})')
ax.set_xlabel('Observed effect size (mean difference)')
ax.set_ylabel('Frequency')
ax.set_title("Winner's Curse: Published studies overestimate the true effect")
ax.legend()
plt.tight_layout()
plt.show()

print(f"True effect size: {true_effect}")
print(f"Mean effect in ALL studies: {np.mean(observed_effects):.3f}")
print(f"Mean effect in PUBLISHED studies: {np.mean(published_effects):.3f}")
print(f"Overestimation factor: {np.mean(published_effects) / true_effect:.1f}x")


### Exercise 6.2

Why is the average published effect size larger than the true effect? What mechanism causes this?

**YOUR ANSWER:**



---
## Part 7 — Extend with AI

### 7.1 Power curve

Use AI (or write code yourself) to answer: **how does BH power depend on effect size?**

Suggested prompt to give your AI assistant (the [UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), Claude, or ChatGPT):

> "Reply with a single Python code block (no prose, no .docx file, no surrounding explanation — just runnable code I can paste into a Colab cell).
>
> Simulate 10,000 genes where 200 are truly alternative with effect size *d*. Vary *d* from 0.5 to 3.0 in steps of 0.25. For each *d*, apply the Benjamini–Hochberg procedure at FDR = 0.10 and compute power as true positives / 200. Plot power versus *d* on a labeled matplotlib figure. Use `numpy.random.seed(2026)`, `n = 25` per group, a two-sample t-test from `scipy.stats`, and `statsmodels.stats.multitest.multipletests` with `method='fdr_bh'`."

**Why the format instruction matters.** Without it, some AI platforms (LibreChat in particular) wrap long answers in a Word document — which you can't paste directly into Colab. Asking explicitly for a code block keeps the output paste-ready.

In [ ]:
# YOUR CODE HERE (AI-generated or your own)


### Exercise 7.1

At what effect size does BH achieve 80% power?

**YOUR ANSWER:**



### 7.2 Critique table

After running the AI-generated power-curve code, paste a brief description of what it produced and complete the table.

**What the AI's code did (one or two sentences):**


| Criterion | Pass / Fail / Partial | Notes |
|---|---|---|
| Did it set `numpy.random.seed(2026)` (or equivalent)? | | |
| Does the simulation match the setup (10,000 genes, 200 alternative, n=25 per group)? | | |
| Did it apply BH correction (`multipletests` with `method='fdr_bh'`), not Bonferroni or no correction? | | |
| Does the plot have axis labels, a title, and a clear power-vs-effect-size trend? | | |
| Does power increase with effect size and approach 1.0 in the large-$d$ limit? | | |
| Are there subtle errors (wrong group sizes, wrong FDR threshold, wrong noise level)? | | |

**One-sentence overall assessment** — would you use this code as-is in a methods section, or did you have to edit it?

---
## Wrap-Up Questions

1. What does the p-value histogram tell you *before* you run any correction? (Three things.)
2. Why is the uncorrected approach dangerous when testing thousands of genes?
3. When would Bonferroni be the right choice over BH?
4. How does publication bias turn individual false positives into a field-wide problem?
5. Looking at the Golub p-value histogram: what questions does it raise that we haven't answered yet?

---
## Looking Ahead

The histogram told us there's signal in the Golub data, and Bonferroni is too blunt to find most of it. In **Lecture 2**, you'll learn how the BH procedure actually works — and what happens when Brad Efron notices that the histogram is "too wide." In **Lab 2**, you'll apply FDR methods, q-values, and the two-groups model to this same dataset.